# Week 8: K-Nearest Neighbors (KNN) and Distance Metrics

## 1. Notebook Setup
- Import libraries

In [ ]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, mean_squared_error

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook", font_scale=1.1)


## 2. Dataset 1: customer_churn
### 2.1 Data Overview & Preparation

In [ ]:
#load data
train_cc = pd.read_csv('customer_churn_dataset-training-master.csv')
test_cc = pd.read_csv('customer_churn_dataset-testing-master.csv')
#combine train and test set
customer_churn = pd.concat([train_cc, test_cc], ignore_index=True)
#drop null rows
customer_churn.dropna(inplace=True)

#preprocessing:
target_col = "Churn"
def preprocess(df, target_col):
    X = pd.get_dummies(df.drop(target_col, axis=1), drop_first=True)
    y = df[target_col]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X, X_scaled, y
X_cc, X_cc_scaled, y_cc = preprocess(customer_churn, 'Churn')

### 2.2 Gradient Boosting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_cc_scaled, y_cc, test_size=0.2, random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0]
}
gb_cv = GridSearchCV(GradientBoostingClassifier(), param_grid, cv=5, n_jobs=-1)
gb_cv.fit(X_train, y_train)

print("Best Parameters:", gb_cv.best_params_)
print("Best CV Score:", gb_cv.best_score_)

Decision Tree Classifier: {'Accuracy': 0.9350072247184339, 'Confusion Matrix': array([[41645,  3336],
       [ 3231, 52830]]), 'Report': '              precision    recall  f1-score   support\n\n         0.0       0.93      0.93      0.93     44981\n         1.0       0.94      0.94      0.94     56061\n\n    accuracy                           0.94    101042\n   macro avg       0.93      0.93      0.93    101042\nweighted avg       0.93      0.94      0.93    101042\n'}


In [ ]:
y_pred = gb_cv.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, cmap='Blues', fmt='g')
plt.title("Confusion Matrix – Gradient Boosting (Customer Churn)")
plt.show()

### 2.3: Learning Rate & Estimator Tradeoff

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.2]
train_scores, test_scores = [], []

for lr in learning_rates:
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, max_depth=3, random_state=42)
    gb.fit(X_train, y_train)
    train_scores.append(gb.score(X_train, y_train))
    test_scores.append(gb.score(X_test, y_test))

plt.plot(learning_rates, train_scores, label="Train Accuracy", marker="o")
plt.plot(learning_rates, test_scores, label="Test Accuracy", marker="s")
plt.title("Effect of Learning Rate on Model Performance (Customer Churn)")
plt.xlabel("Learning Rate")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### 2.4: Feature Importance

In [ ]:
best_model = gb_cv.best_estimator_
feature_importances = pd.Series(best_model.feature_importances_, index=pd.get_dummies(customer_churn.drop('Churn', axis=1)).columns)
top_features = feature_importances.sort_values(ascending=True).tail(10)

top_features.plot(kind='barh', color='seagreen')
plt.title("Top 10 Feature Importances – Gradient Boosting (Customer Churn)")
plt.xlabel("Importance Score")
plt.show()

#### Additional plots:
[insert any plots]

## 3. Dataset 2: digital_marketing_campaign

In [ ]:
digital_marketing = pd.read_csv("digital_marketing_campaign_dataset.csv")
X = pd.get_dummies(digital_marketing.drop("Conversion", axis=1, errors="ignore"), drop_first=True)
y = digital_marketing["Conversion"] if "Conversion" in digital_marketing.columns else None

X_dm, X_dm_scaled, y_dm = preprocess(digital_marketing, 'Response')

### 3.1 Gradient Boosting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_dm_scaled, y_dm, test_size=0.2, random_state=42)
param_grid_reg = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0],
    'loss': ['squared_error', 'absolute_error']
}
gb_reg_cv = GridSearchCV(GradientBoostingRegressor(), param_grid_reg, cv=5, n_jobs=-1)
gb_reg_cv.fit(X_train, y_train)

print("Best Parameters:", gb_reg_cv.best_params_)
print("Best CV Score:", gb_reg_cv.best_score_)

Decision Tree Classifier: {'Accuracy': 0.8725, 'Confusion Matrix': array([[ 105,   89],
       [ 115, 1291]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.48      0.54      0.51       194\n           1       0.94      0.92      0.93      1406\n\n    accuracy                           0.87      1600\n   macro avg       0.71      0.73      0.72      1600\nweighted avg       0.88      0.87      0.88      1600\n'}
Random Forest Classifier: {'Accuracy': 0.888125, 'Confusion Matrix': array([[  34,  160],
       [  19, 1387]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.64      0.18      0.28       194\n           1       0.90      0.99      0.94      1406\n\n    accuracy                           0.89      1600\n   macro avg       0.77      0.58      0.61      1600\nweighted avg       0.87      0.89      0.86      1600\n'}


In [ ]:
y_pred = gb_reg_cv.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"Test RMSE: {rmse:.4f}")

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.6)
plt.title("Predicted vs Actual – Gradient Boosting Regression (Marketing Campaign)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.show()

### 3.3: Learning Rate & Estimator Tradeoff

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.2]
train_scores, test_scores = [], []

for lr in learning_rates:
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, max_depth=3, random_state=42)
    gb.fit(X_train, y_train)
    train_scores.append(gb.score(X_train, y_train))
    test_scores.append(gb.score(X_test, y_test))

plt.plot(learning_rates, train_scores, label="Train Accuracy", marker="o")
plt.plot(learning_rates, test_scores, label="Test Accuracy", marker="s")
plt.title("Effect of Learning Rate on Model Performance (Customer Churn)")
plt.xlabel("Learning Rate")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### 3.4: Feature Importance

In [ ]:
best_model = gb_cv.best_estimator_
feature_importances = pd.Series(best_model.feature_importances_, index=pd.get_dummies(customer_churn.drop('Churn', axis=1)).columns)
top_features = feature_importances.sort_values(ascending=True).tail(10)

top_features.plot(kind='barh', color='seagreen')
plt.title("Top 10 Feature Importances – Gradient Boosting (Customer Churn)")
plt.xlabel("Importance Score")
plt.show()

## 4. Dataset 3: marketing_campaign

In [ ]:
marketing_campaign = pd.read_csv("marketing_campaign.csv", sep=';')
X = pd.get_dummies(marketing_campaign.drop("Response", axis=1, errors="ignore"), drop_first=True)
y = marketing_campaign["Response"] if "Response" in marketing_campaign.columns else None

X_mc, X_mc_scaled, y_mc = preprocess(marketing_campaign, 'Success')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_mc_scaled, y_mc, test_size=0.2, random_state=42)
param_grid_reg = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0],
    'loss': ['squared_error', 'absolute_error']
}
gb_reg_cv = GridSearchCV(GradientBoostingRegressor(), param_grid_reg, cv=5, n_jobs=-1)
gb_reg_cv.fit(X_train, y_train)

print("Best Parameters:", gb_reg_cv.best_params_)
print("Best CV Score:", gb_reg_cv.best_score_)

In [ ]:
y_pred = gb_reg_cv.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"Test RMSE: {rmse:.4f}")

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.6)
plt.title("Predicted vs Actual – Gradient Boosting Regression (Marketing Campaign)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.show()

### 4.3: Learning Rate & Estimator Tradeoff

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.2]
train_scores, test_scores = [], []

for lr in learning_rates:
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, max_depth=3, random_state=42)
    gb.fit(X_train, y_train)
    train_scores.append(gb.score(X_train, y_train))
    test_scores.append(gb.score(X_test, y_test))

plt.plot(learning_rates, train_scores, label="Train Accuracy", marker="o")
plt.plot(learning_rates, test_scores, label="Test Accuracy", marker="s")
plt.title("Effect of Learning Rate on Model Performance (Customer Churn)")
plt.xlabel("Learning Rate")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### 4.4: Feature Importance

In [ ]:
best_model = gb_cv.best_estimator_
feature_importances = pd.Series(best_model.feature_importances_, index=pd.get_dummies(customer_churn.drop('Churn', axis=1)).columns)
top_features = feature_importances.sort_values(ascending=True).tail(10)

top_features.plot(kind='barh', color='seagreen')
plt.title("Top 10 Feature Importances – Gradient Boosting (Customer Churn)")
plt.xlabel("Importance Score")
plt.show()